https://www.kaggle.com/datasets/arnabchaki/data-science-salaries-2025

About Dataset
Data Science Job Salaries Dataset contains 11 columns, each are:

work_year: The year the salary was paid.
experience_level: The experience level in the job during the year
employment_type: The type of employment for the role
job_title: The role worked in during the year.
salary: The total gross salary amount paid.
salary_currency: The currency of the salary paid as an ISO 4217 currency code.
salaryinusd: The salary in USD
employee_residence: Employee's primary country of residence in during the work year as an ISO 3166 country code.
remote_ratio: The overall amount of work done remotely
company_location: The country of the employer's main office or contracting branch
company_size: The median number of people that worked for the company during the year

In [20]:
import pandas as pd

In [21]:
df = pd.read_csv('salaries.csv')
df.head()


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2025,EN,FT,Data Analist,65664,EUR,69120,NL,0,NL,M
1,2025,EN,FT,Data Analist,47652,EUR,50160,NL,0,NL,M
2,2025,EN,FT,Data Engineer,158113,USD,158113,US,0,US,M
3,2025,EN,FT,Data Engineer,87795,USD,87795,US,0,US,M
4,2025,EX,FT,Data Engineer,351410,USD,351410,US,0,US,M


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105434 entries, 0 to 105433
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   work_year           105434 non-null  int64 
 1   experience_level    105434 non-null  object
 2   employment_type     105434 non-null  object
 3   job_title           105434 non-null  object
 4   salary              105434 non-null  int64 
 5   salary_currency     105434 non-null  object
 6   salary_in_usd       105434 non-null  int64 
 7   employee_residence  105434 non-null  object
 8   remote_ratio        105434 non-null  int64 
 9   company_location    105434 non-null  object
 10  company_size        105434 non-null  object
dtypes: int64(4), object(7)
memory usage: 8.8+ MB


In [23]:
df.nunique()

work_year                 6
experience_level          4
employment_type           4
job_title               347
salary                 9476
salary_currency          26
salary_in_usd         10476
employee_residence       98
remote_ratio              3
company_location         92
company_size              3
dtype: int64

In [24]:
df.shape

(105434, 11)

так как в нашем датафрейме сразу есть конвертация валюты зарплаты в доллары, мы модем дропнуть не нужные нам для анализа колонки

In [25]:
df_cleaned = df.copy()
df_cleaned = df_cleaned.drop(["salary", "salary_currency"], axis=1)

In [26]:
df_cleaned.head()

,work_year,experience_level,employment_type,job_title,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2025,EN,FT,Data Analist,69120,NL,0,NL,M
1,2025,EN,FT,Data Analist,50160,NL,0,NL,M
2,2025,EN,FT,Data Engineer,158113,US,0,US,M
3,2025,EN,FT,Data Engineer,87795,US,0,US,M
4,2025,EX,FT,Data Engineer,351410,US,0,US,M


In [27]:
df_cleaned = df_cleaned.drop_duplicates() # дропнем все дубликаты 

In [28]:
df_cleaned.isnull().sum() # пропущенных значений, к счастью, нет

work_year             0
experience_level      0
employment_type       0
job_title             0
salary_in_usd         0
employee_residence    0
remote_ratio          0
company_location      0
company_size          0
dtype: int64

In [29]:
def summarize_job_title(title):
    title = title.lower()
    if 'data scientist' in title or 'science' in title or 'nlp' in title or 'research' in title:
        return 'Data Science'
    elif 'analyst' in title or 'analytics' in title or 'bi ' in title or 'business intelligence' in title:
        return 'Data Analytics'
    elif 'engineer' in title or 'etl' in title or 'infrastructure' in title or 'architect' in title:
        return 'Data Engineering'
    elif 'machine learning' in title or 'ml' in title or 'ai ' in title or 'computer vision' in title:
        return 'Machine Learning'
    elif 'manager' in title or 'head' in title or 'lead' in title or 'director' in title or 'officer' in title:
        return 'Management/Leadership'
    else:
        return 'Other'

In [30]:
df_cleaned['job_category'] = df_cleaned['job_title'].apply(summarize_job_title)
print(df_cleaned['job_category'].value_counts())

job_category
Data Engineering         21601
Data Analytics           11352
Data Science              8506
Other                     5201
Management/Leadership     5000
Machine Learning           758
Name: count, dtype: int64


In [31]:
# Смотрим топ-20 названий, которые ушли в категорию Other
print(df_cleaned[df_cleaned['job_category'] == 'Other']['job_title'].value_counts().head(10))

job_title
Associate                     890
Consultant                    507
Data Specialist               472
Applied Scientist             467
Software Developer            394
Developer                     380
Data Governance               167
Data Management Specialist    135
Decision Scientist            116
Data Modeler                  103
Name: count, dtype: int64


In [32]:
df.head()

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2025,EN,FT,Data Analist,65664,EUR,69120,NL,0,NL,M
1,2025,EN,FT,Data Analist,47652,EUR,50160,NL,0,NL,M
2,2025,EN,FT,Data Engineer,158113,USD,158113,US,0,US,M
3,2025,EN,FT,Data Engineer,87795,USD,87795,US,0,US,M
4,2025,EX,FT,Data Engineer,351410,USD,351410,US,0,US,M


In [33]:
# реализуем функцию, которая будет определять уровень зарплаты, основываясь на стране, в которой находится вакансия

def get_salary_level(x):
    lower_boundary = x.quantile(0.33) 
    upper_boundary = x.quantile(0.66)

    try:
            return pd.cut(x, 
                        bins=[-float('inf'), lower_boundary, upper_boundary, float('inf')], 
                        labels=['Low', 'Medium', 'High'],
                        duplicates='drop')
    except ValueError:
        return "Medium"

In [34]:
df_cleaned['salary_level'] = df_cleaned.groupby('company_location')['salary_in_usd'].transform(get_salary_level)
print(df_cleaned.groupby(['company_location', 'salary_level'], observed=False)['salary_in_usd'].mean().head(10))

company_location  salary_level
AD                Medium           50745.000000
AE                High            117500.000000
                  Low              52500.000000
                  Medium           95000.000000
AM                High             85333.333333
                  Low              21333.000000
                  Medium           48100.000000
AR                High            131888.222222
                  Low              35623.529412
                  Medium           71308.823529
Name: salary_in_usd, dtype: float64


In [35]:
# Создаем колонку: True, если человек работает на зарубежную компанию
df_cleaned['is_remote_nomad'] = df_cleaned['employee_residence'] != df_cleaned['company_location']
print(df_cleaned['is_remote_nomad'].value_counts(normalize=True))

is_remote_nomad
False    0.997177
True     0.002823
Name: proportion, dtype: float64


In [36]:
analysis_result = df_cleaned.groupby(['job_category', 'salary_level'], observed=False)['salary_in_usd'].mean().unstack()
print(analysis_result)

salary_level                    High           Low         Medium
job_category                                                     
Data Analytics         206456.061958  80854.803229  135053.426479
Data Engineering       233524.179321  88772.163944  140821.040258
Data Science           235460.310172  83407.431373  140360.517988
Machine Learning       236743.777090  81252.478947  138150.248980
Management/Leadership  233580.184988  89807.942993  140309.218301
Other                  220913.137730  80584.453947  139104.222788


In [37]:
df_cleaned.head()

,work_year,experience_level,employment_type,job_title,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,job_category,salary_level,is_remote_nomad
0,2025,EN,FT,Data Analist,69120,NL,0,NL,M,Other,Medium,False
1,2025,EN,FT,Data Analist,50160,NL,0,NL,M,Other,Low,False
2,2025,EN,FT,Data Engineer,158113,US,0,US,M,Data Engineering,Medium,False
3,2025,EN,FT,Data Engineer,87795,US,0,US,M,Data Engineering,Low,False
4,2025,EX,FT,Data Engineer,351410,US,0,US,M,Data Engineering,High,False


In [38]:
df_cleaned.to_csv('final_salaries_cleaned.csv', index=False)

print("Файл успешно сохранен!")

Файл успешно сохранен!
